# 🎬 Hoichoi Problem 2 — Live GPU ML Inference Server
### Turn Google Colab's Free T4 GPU into a High-Speed REST Backend for your Vercel Web App

This notebook runs the complete **Problem 2 Pipeline Server** (`ml_server.py`):
- 🧠 **ASR Model**: `SayedShaun/bengali-whisper-medium` + TIES-Merged LoRA (`base` + `mid` + `high`)
- 🎙️ **VAD & Anti-Hallucination Guard**: Silero-VAD (`snakers4/silero-vad`)
- 👥 **Acoustic Speaker Diarization**: Persistent speaker clustering
- 🌐 **Translation**: Helsinki-NLP `opus-mt-bn-en` + Hindi
- 📐 **Timed-Text Formatting**: Netflix / BBC specs (CPS & Line Limits)
- 🛡️ **10-Point QC Auditor**: Ranked human review queue
- ⚡ **Speed**: ~25x faster than real-time on Colab T4 GPU (processes a 23-minute episode in under 1 minute!)

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone Repository
!git clone https://github.com/Dustu103/vvd_Asir.git hoichoi
%cd hoichoi

In [ ]:
# 3. Install Dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers peft accelerate soundfile librosa pyngrok gdown scikit-learn
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
# 4. Download Trained Adapters from Google Drive
!python -c "import gdown, os; gdown.download_folder(id='1-VKKplYGNKywkcj7rGbV3cQDqAdgEVBw', output='adapters/merged', quiet=False) if not os.path.exists('adapters/merged/adapter_model.safetensors') else None"

In [ ]:
# 5. Expose Public Tunnel via pyngrok or localtunnel
# Enter your free ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""  # Optional: paste token here

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000).public_url
    print(f"\n🚀 YOUR PUBLIC ML BACKEND URL: {public_url}")
    print(f"👉 Paste this URL into your Vercel Web App or set ML_BACKEND_URL in Vercel settings!")
else:
    # Fallback to localtunnel
    !npm install -g localtunnel > /dev/null 2>&1
    import subprocess, time
    lt = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE, text=True)
    time.sleep(3)
    print("Tunnel started. Output will appear below:")

In [ ]:
# 6. Start the High-Speed GPU ML Model Server
!python ml_server.py